<a href="https://colab.research.google.com/github/mayait/CursoAnalisisDatos_IA_2026/blob/main/sitio/labs/lab_02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# Laboratorio 2 · La pregunta antes del dato

La semana pasada viste un informe con datos correctos y conclusión inválida: el error estaba en la
comparación, no en el cálculo. Hoy retrocedes un paso más, hasta el punto donde ese error nació.
Antes de abrir un archivo hay cuatro cosas que decidir —unidad de análisis, métrica, comparación y
decisión— y si falta una, el código que escribas después no la va a salvar. Hoy desbloqueas la carga
de datos en serio (separador, codificación y fechas) y la única pregunta que separa a un analista de
un generador de reportes: ¿qué decisión cambia según el resultado?

> **Hoy haces** · Cargas la base del curso con los parámetros correctos y la auditas con cuatro
> métodos (90 min). Después demuestras, con el mismo archivo, que «una fila», «una venta» y «un
> cliente» son tres cosas distintas y producen tres respuestas distintas a la misma pregunta.
> Cierras redactando la ficha de análisis de tu caso.
>
> **Entrega** · Este cuaderno ejecutado, los tres ejercicios resueltos y la ficha de análisis del
> proyecto de tu equipo completa en la última celda. Nombre de archivo: `lab_02_apellido.ipynb`.

In [ ]:
# --- Setup del entorno ---
from pathlib import Path
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (10, 4)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

# Los datos de Comercial Andina viven en sitio/datos/
REPO = "https://github.com/mayait/CursoAnalisisDatos_IA_2026.git"
COPIA = Path("/content/CursoAnalisisDatos_IA_2026")
CANDIDATOS = [Path("../datos"), Path("datos"), Path("sitio/datos"),
              COPIA / "sitio" / "datos"]
DATOS = next((p for p in CANDIDATOS if p.exists()), None)
if DATOS is None:
    # En Colab el cuaderno llega solo: se trae el repositorio una sola vez.
    import subprocess
    subprocess.run(["git", "clone", "--depth", "1", REPO, str(COPIA)], check=True)
    DATOS = COPIA / "sitio" / "datos"

print("Setup completo ✓")
print(f"pandas {pd.__version__} · datos en {DATOS.resolve()}")

## 1. Las cuatro preguntas que se contestan antes de abrir el archivo

Una petición de reporte suena así: *«pásame las ventas de Cuenca»*. Una pregunta de negocio suena
así: *«¿le damos a Cuenca el presupuesto comercial que le dimos a Quito?»*. La primera se contesta
con una consulta; la segunda exige decidir cuatro cosas antes de escribir una línea de código.

Si no puedes llenar las cuatro filas de la tabla siguiente, todavía no tienes un análisis: tienes
una consulta.

In [ ]:
ficha = pd.DataFrame([
    ("Unidad de análisis", "¿Qué representa una fila del resultado?",
     "Un cliente (no una línea de factura, no una factura)"),
    ("Métrica", "¿Qué número mide exactamente eso?",
     "Facturación acumulada por cliente en los últimos 12 meses"),
    ("Comparación", "¿Comparado con qué?",
     "El mismo indicador en Quito, en el mismo periodo"),
    ("Decisión", "¿Qué cambia según el resultado?",
     "Si Cuenca supera a Quito, se le asigna un vendedor más el próximo trimestre"),
], columns=["elemento", "pregunta", "respuesta para este caso"])
ficha

📌 La comparación es la que más se olvida y la que más duele. Un número solo —«Cuenca facturó
484 426,89»— no es ni bueno ni malo hasta que dices contra qué. La semana 9 vuelve sobre esto con
estadística; hoy basta con no dejar la fila vacía.

## 2. Cargar un archivo de verdad: separador, codificación y fechas

`pd.read_csv("archivo.csv")` funciona el 60 % de las veces. El otro 40 % son archivos exportados por
un ERP en Ecuador, con punto y coma por separador, codificación latina y fechas en formato local.
Los tres fallos, uno por uno.

**Separador.** Si te equivocas, pandas no falla: te devuelve una tabla de una sola columna.

In [ ]:
mal_separado = pd.read_csv(DATOS / "clientes.csv", sep=";")

print(f"forma con sep=';'  : {mal_separado.shape}")
print(f"columnas leídas    : {list(mal_separado.columns)}")
mal_separado.head(3)

Seis columnas se convirtieron en una. Nadie recibió un error: si esto ocurre dentro de un proceso
automático, el análisis sigue corriendo con basura.

**Codificación.** El archivo está en UTF-8. Leerlo como latino no rompe nada; solo destruye las
tildes, y las tildes son parte de los datos cuando la columna es una ciudad.

In [ ]:
utf8 = pd.read_csv(DATOS / "clientes.csv", encoding="utf-8")
latin = pd.read_csv(DATOS / "clientes.csv", encoding="latin-1")

print("con encoding='utf-8'   :", [c for c in utf8["ciudad"].unique() if "í" in c])
print("con encoding='latin-1' :", [c for c in latin["ciudad"].unique() if "Ã" in c])
print()
print(f"filas donde las dos lecturas difieren : {(utf8['ciudad'] != latin['ciudad']).sum()}")
print("son pocas, y son justo las que un groupby por ciudad va a dejar sueltas")

**Fechas.** `parse_dates` es el parámetro que más confianza falsa genera. Con `clientes.csv`
funciona: convierte la columna y respeta los ausentes.

In [ ]:
clientes = pd.read_csv(DATOS / "clientes.csv", encoding="utf-8", parse_dates=["fecha_alta"])

print(f"tipo de fecha_alta : {clientes['fecha_alta'].dtype}")
print(f"ausentes           : {clientes['fecha_alta'].isna().sum()} de {len(clientes)} "
      f"({clientes['fecha_alta'].isna().mean():.1%})")
print(f"rango              : {clientes['fecha_alta'].min():%d-%m-%Y} a {clientes['fecha_alta'].max():%d-%m-%Y}")

Ahora el mismo parámetro sobre `ventas.csv`, donde las fechas vienen en dos formatos mezclados.

In [ ]:
ventas = pd.read_csv(DATOS / "ventas.csv", sep=",", encoding="utf-8", parse_dates=["fecha"])

print(f"tipo de fecha : {ventas['fecha'].dtype}")
print("primeras filas:", ventas["fecha"].head(4).tolist())
print()
print("¿puedo hacer aritmética de fechas?")
try:
    print(ventas["fecha"].max() - ventas["fecha"].min())
except TypeError as e:
    print(f"  no: {e}")

⚠️ `parse_dates` **no lanzó ningún error**: devolvió la columna como texto y siguió. El tipo quedó en
`object` y cualquier cálculo de antigüedad, recencia o estacionalidad que hagas después estará
comparando cadenas de caracteres. Este es el fallo silencioso más caro de pandas y la razón por la
que `dtypes` es el primer método que se mira, no el último.

La solución de hoy es de una línea; el diagnóstico completo de por qué el archivo llegó así y qué se
pierde al arreglarlo es la semana 4.

In [ ]:
ventas["fecha"] = pd.to_datetime(ventas["fecha"], format="mixed", dayfirst=True)

print(f"tipo de fecha : {ventas['fecha'].dtype}")
print(f"rango         : {ventas['fecha'].min():%d-%m-%Y} a {ventas['fecha'].max():%d-%m-%Y}")
print(f"periodo       : {(ventas['fecha'].max() - ventas['fecha'].min()).days} días")

## 3. Los cinco minutos: `shape`, `head`, `info` y `sample`

Cuatro métodos, en este orden, siempre, antes de cualquier otra cosa. Cada uno contesta una pregunta
distinta y ninguno sustituye a los otros.

In [ ]:
print(f"shape : {ventas.shape[0]:,} filas × {ventas.shape[1]} columnas")
print(f"desde : {ventas['fecha'].min():%d-%m-%Y}   hasta: {ventas['fecha'].max():%d-%m-%Y}")
ventas.head()

In [ ]:
ventas.info()

`info()` es el más denso de los cuatro. Léelo así: **80 515 filas**, pero `cliente_id` solo tiene
74 242 valores no nulos. Faltan 6 273 identificadores, un 7,8 % de las líneas. Ese hueco no se ve en
`head()` porque las primeras cinco filas están completas: por eso existe `sample()`.

In [ ]:
ventas.sample(8, random_state=SEED)

`head()` te muestra el principio del archivo, que suele estar ordenado y limpio. `sample()` te
muestra el archivo de verdad: en esta muestra de ocho filas ya aparece un `cliente_id` vacío. Cambia
el `random_state` y ejecútala tres o cuatro veces más: van a salir cantidades negativas y facturas
que empiezan con `C` en lugar de `F`. Anota tres observaciones antes de seguir.

## 4. La unidad de análisis: línea ≠ factura ≠ cliente

Esta es la sección de la semana. El gerente pregunta *«¿cuál es nuestro ticket promedio?»* y la
respuesta depende por completo de qué llames «una venta». El mismo archivo, tres unidades, tres
números.

In [ ]:
print(f"filas del archivo (líneas de factura) : {len(ventas):,}")
print(f"facturas distintas                    : {ventas['factura_id'].nunique():,}")
print(f"clientes distintos con compras        : {ventas['cliente_id'].nunique():,}")
print(f"clientes en el padrón                 : {len(clientes):,}")

In [ ]:
ventas["monto"] = ventas["cantidad"] * ventas["precio_unitario"] * (1 - ventas["descuento"])

por_factura = ventas.groupby("factura_id")["monto"].sum()
por_cliente = ventas.dropna(subset=["cliente_id"]).groupby("cliente_id")["monto"].sum()

niveles = pd.DataFrame({
    "unidad de análisis": ["línea de factura", "factura", "cliente"],
    "cuántas hay": [len(ventas), len(por_factura), len(por_cliente)],
    "monto medio": [ventas["monto"].mean(), por_factura.mean(), por_cliente.mean()],
    "qué pregunta contesta": [
        "¿cuánto pesa un producto dentro del carrito?",
        "¿cuánto gasta alguien cuando viene a comprar?",
        "¿cuánto vale un cliente para la empresa?",
    ],
})
niveles

📌 **35,43 · 160,76 · 1 502,10.** Los tres son «el promedio de ventas» y los tres son correctos. El
que el gerente quería cuando dijo «ticket promedio» es el segundo. Si le entregas el primero, la
cifra queda 4,5 veces más baja; si le entregas el tercero, 9,3 veces más alta. Ninguna de las tres
está mal calculada: dos están mal elegidas.

Los factores que separan los tres niveles no son constantes, y por eso no puedes convertir uno en
otro con una regla de tres.

In [ ]:
lineas_por_factura = ventas.groupby("factura_id").size()
facturas_por_cliente = ventas.dropna(subset=["cliente_id"]).groupby("cliente_id")["factura_id"].nunique()

resumen = pd.DataFrame({
    "líneas por factura": lineas_por_factura.describe()[["mean", "50%", "min", "max"]],
    "facturas por cliente": facturas_por_cliente.describe()[["mean", "50%", "min", "max"]],
})
resumen.index = ["media", "mediana", "mínimo", "máximo"]
resumen

Una factura tiene entre 1 y 13 líneas; un cliente tiene entre 1 y 72 facturas. Un cliente con 72
facturas pesa 72 veces más que uno con una sola cuando promedias por línea, y exactamente igual
cuando promedias por cliente. Elegir la unidad de análisis **es** elegir a quién le das voz.

## 5. Clínica de definiciones: ¿cuántos clientes activos tenemos?

Pregunta de directorio, aparentemente trivial. Nadie definió «activo». Cada definición razonable da
un número distinto, y todos son defendibles.

In [ ]:
ultima = ventas["fecha"].max()
ultima_compra = ventas.dropna(subset=["cliente_id"]).groupby("cliente_id")["fecha"].max()

definiciones = [(f"compró en los últimos {d} días", int((ultima_compra >= ultima - pd.Timedelta(days=d)).sum()))
                for d in (30, 90, 180, 365)]
definiciones += [("compró alguna vez", int(ultima_compra.size)),
                 ("está dado de alta en el padrón", len(clientes))]

activos = pd.DataFrame(definiciones, columns=["definición de cliente activo", "cuántos son"])
activos["% del padrón"] = activos["cuántos son"] / len(clientes) * 100

fig, ax = plt.subplots(figsize=(10, 3.4))
ax.barh(activos["definición de cliente activo"], activos["cuántos son"], color="#4C72B0")
ax.invert_yaxis()
ax.set_title("La misma pregunta admite seis respuestas: de 233 a 1 800 clientes activos")
ax.set_xlabel("clientes activos")
for y, valor in enumerate(activos["cuántos son"]):
    ax.text(valor + 25, y, f"{valor:,}", va="center")
plt.tight_layout()
plt.show()

activos

De 233 a 1 800: un factor de **7,7**. Si el directorio pide «el número de clientes activos» y tú
eliges la definición sin decirlo, estás decidiendo tú el titular de la reunión. La regla del curso:
la definición se escribe **antes** de calcular, se pone al lado del número y se mantiene igual todos
los meses. Un indicador cuya definición cambia entre trimestres no es un indicador.

## 6. La ficha de análisis

Una página, sin código, antes de tocar el teclado. Es lo que entregas a otro grupo para que la
ataque. Copia esta plantilla a una celda de texto nueva y llénala con el caso de tu equipo.

---

### Ficha de análisis · *nombre del caso*

| Campo | Contenido |
|---|---|
| **Pregunta de negocio** | *Una frase, en interrogativa, que un gerente haría en voz alta.* |
| **Unidad de análisis** | *Qué representa una fila del resultado.* |
| **Métrica** | *La fórmula exacta, con periodo y filtros.* |
| **Comparación** | *Contra qué: otro periodo, otro grupo, una meta.* |
| **Decisión que cambia** | *Qué se hace distinto según el resultado. Si no cambia nada, cambia la pregunta.* |
| **Fuente y granularidad** | *Qué archivo, qué representa una fila del origen.* |
| **Limitaciones conocidas** | *Qué no puedes contestar con estos datos.* |

---

Y la misma ficha, ejecutable, para que quede dentro del cuaderno entregado.

In [ ]:
mi_ficha = pd.Series({
    "pregunta_de_negocio": "¿Cuántos de nuestros clientes dejaron de comprar y cuánto nos costó?",
    "unidad_de_analisis": "Un cliente",
    "metrica": "Días desde la última compra; facturación de los 12 meses previos a esa fecha",
    "comparacion": "Clientes con última compra hace más de 180 días frente al resto",
    "decision_que_cambia": "A quién llama el equipo de televentas la próxima semana y con qué oferta",
    "fuente_y_granularidad": "ventas.csv, una fila = una línea de factura; clientes.csv, una fila = un cliente",
    "limitaciones_conocidas": "El 7,8 % de las líneas no tiene cliente_id: esas ventas no se pueden atribuir",
})
mi_ficha.to_frame("contenido")

### 🌶️ Ejercicio 1 — Guiado

Carga `productos.csv` y `sucursales.csv` y aplícales los cinco minutos completos: `shape`, `head`,
`info` y `sample`. Después responde en una frase, para cada archivo, **qué representa una fila**.

In [ ]:
# TU CÓDIGO AQUÍ
# Pista: son archivos pequeños; con sucursales.csv el head() es el archivo entero.
# Fíjate en la fila S99 antes de escribir qué representa una fila de sucursales.csv.

### 🔥 Desafío

La columna `factura_id` tiene dos prefijos: `F` y `C`. Averigua qué son las que empiezan con `C`,
cuántas hay y qué relación tienen con la columna `es_devolucion`. Después responde: cuando contaste
17 675 «facturas distintas» en la sección 4, ¿contaste ventas o contaste documentos? Corrige el
número de facturas de venta.

In [ ]:
# TU CÓDIGO AQUÍ
# Pista 1: ventas["factura_id"].str[0].value_counts()
# Pista 2: cruza ese prefijo con ventas["es_devolucion"] usando pd.crosstab

### 🎯 Reto en clase (15 min)

En equipos. Escriban la ficha de análisis de su caso —las siete filas, sin saltarse ninguna— y
entréguensela a otro equipo. El equipo receptor tiene cinco minutos para atacar dos cosas: la
**unidad de análisis** y la fila de **decisión que cambia**. Si logran demostrar que la respuesta no
cambia ninguna decisión, la ficha vuelve al remitente y se reescribe.

In [ ]:
# TU CÓDIGO AQUÍ
# Pista: copia la estructura de mi_ficha y reemplaza los siete valores por los de tu caso.
# Deja la ficha impresa en el cuaderno: es parte del entregable.

## La trampa de hoy

⚠️ **Empezar a programar antes de poder decir en una frase qué decisión cambia según el resultado.**
Se reconoce porque el análisis termina en un número que nadie sabe si es bueno. El directorio de
Comercial Andina encargó dimensionar el equipo de televentas para el próximo trimestre: un asesor
por cada 60 clientes activos. Dos analistas atacaron el mismo archivo el mismo día.

In [ ]:
CARTERA_POR_ASESOR = 60

analista_a = int((ultima_compra >= ultima - pd.Timedelta(days=30)).sum())   # "activo = compró este mes"
analista_b = int((ultima_compra >= ultima - pd.Timedelta(days=365)).sum())  # "activo = compró este año"

trampa = pd.DataFrame({
    "definición usada": ["compró en los últimos 30 días", "compró en los últimos 365 días"],
    "clientes activos": [analista_a, analista_b],
    "asesores a contratar": [np.ceil(analista_a / CARTERA_POR_ASESOR).astype(int),
                             np.ceil(analista_b / CARTERA_POR_ASESOR).astype(int)],
})
print(trampa.to_string(index=False))
print()
print(f"Misma pregunta, mismo archivo, mismo día: {trampa['asesores a contratar'][1] - trampa['asesores a contratar'][0]} "
      f"asesores de diferencia ({trampa['asesores a contratar'][1] / trampa['asesores a contratar'][0]:.1f} veces).")

Ninguno de los dos se equivocó al programar. Los dos escribieron el `groupby` correcto y el filtro
correcto. La diferencia son 22 contrataciones y aparece **antes** de la primera línea de código, en
una definición que nadie escribió. Por eso la ficha de análisis va primero: no es burocracia, es la
única parte del trabajo que el asistente de IA no puede hacer por ti, porque depende de una decisión
de negocio que no está en los datos.

## Entregable

Sube `lab_02_apellido.ipynb` con:

- El cuaderno ejecutado de arriba a abajo, con la carga hecha con separador, codificación y fechas
  explícitos.
- Los tres ejercicios resueltos. En el desafío, el número corregido de facturas de venta.
- Tres observaciones escritas que salgan de `info()` y `sample()`, con la cifra que las respalda.
- La ficha de análisis de tu equipo, completa, en la última celda.
- Una fila nueva en la bitácora de prompts: pídele al asistente que critique tu ficha como si fuera
  el gerente que va a recibir el resultado y anota qué objeciones aceptaste y cuáles no.

## Para tu equipo

- La empresa del proyecto se elige esta semana y no se cambia. Antes de decidir, comprueben que
  pueden conseguir un archivo real con al menos mil filas.
- La ficha de análisis es el primer entregable del proyecto integrador. La fila que más les va a
  costar es «decisión que cambia»: si no la pueden llenar, el caso no sirve, por interesante que sea.
- Anoten desde ya cuál es la unidad de análisis de su archivo y si coincide con la unidad de la
  pregunta. En la mayoría de los casos reales no coincide, y ahí es donde la semana 5 les va a hacer
  falta.